# Generating YOLO labels

We will store metadata for our files in `/data/data.yaml`.  This file will help with training.

YOLO expects each training image file to have a corresponding `txt` file with the class information and bounding boxes.

As I've taken the approach of generating training images synthetically per class, each image will be centered around our target molecule. The width and height of the bounding box can be the same as the image itself.  

Class order must be maintained between model versions if weights are to be persisted so I will use an alphabetically-ordered list of each atom as my class list:

`classes = ["B", "Br", "C","Cl","F","I","N","O","P","S","Si"]`


Each line in a YOLO label file looks like this:

`<class_id> <x_center> <y_center> <width> <height>`

All values after `class_id` are normalized (0 to 1) relative to image size.  To use the whole image, centred on the atom we can use `<class_id> 0.5 0.5 1 1`.

Each image file needs a corresponding label file. In [the previous notebook](notebooks/03_generating_training_images.ipynb) we generated 20K small images for each class and stored them in 20 subfolders for each atom (1,000 images in each) under the following layout:

```
data/
├── images/
│   ├── train_synthetic/
│       └── C/
│           └── 00/
│               └── C_00001.png
                    ...
                    ...
│               └── C_00999.png                 
│           └── 20/
│               └── C_20000.png
```

Where `C/` is the parent folder for carbon, and similar top level folders exist for each of the other elements.  

We therefore need an analagous layout for the label text files:

```
data/
├── labels/
│   ├── train_synthetic/
│       └── C/
│           └── 00/
│               └── C_00001.txt
                    ...
                    ...
```

I've made life a little harder for myself as I am writing the labels _after_ I have generated the images. In hindsight I should have done this at the same time.

The number of training Images vaires slightly depending on the atom, it isn't exactly but _at least_ 20,000 images.

One way to generate the labels would be to find all images files in the training directoy for a given atom, replace their path string to be `labels` instead of `images`, make them txt files and change their contents.

We can replace the path string for the image and replace a few substrings to get the path of the labels file:

`mol-translation/data/images/train_synthetic/C/00/C_00001.png` becomes 

`mol-translation/data/labels/train_synthetic/C/00/C_00001.txt` 

We can therefore replace `images` to `labels`, and `png` to `txt`.


In [42]:
from pathlib import Path

classes = ["B", "Br", "C","Cl","F","I","N","O","P","S","Si"]

images_base_dir = Path(f"/mol-translation/data/images/train_synthetic/")

replacements = {
    "images": "labels",
    "png": "txt",
}

def replace_all_paths(p, replacements):
    s = str(p)
    for old, new in replacements.items():
        s = s.replace(old, new)
    return Path(s)

def write_yolo_txt(p, txt):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(txt, encoding="utf-8")
    

for i,c in enumerate(classes):

    print(f"Preparing labels for element: {c}")
    yolo_txt = f"{i} 0.5 0.5 1 1"
    
    directory = images_base_dir / c
    files = [f for f in directory.rglob("*") if f.is_file()]
    new_files = [replace_all_paths(f, replacements) for f in files]

    for new_file in new_files:
        write_yolo_txt(new_file, yolo_txt)


Preparing labels for element: B
Preparing labels for element: Br
Preparing labels for element: C
Preparing labels for element: Cl
Preparing labels for element: F
Preparing labels for element: I
Preparing labels for element: N
Preparing labels for element: O
Preparing labels for element: P
Preparing labels for element: S
Preparing labels for element: Si


The above executed in a matter of seconds, far faster than generating the images themselves!

Now we have labels and images, we can try and train a model!